### 1. Import Libraries

In [14]:
import os
import pandas as pd

from statsmodels.tsa.stattools import adfuller

### 2. Reproducible Data Preprocessing Pipeline

To ensure reproducibility, all preprocessing steps are encapsulated in a function. This allows the same transformations to be applied consistently across different runs and datasets.

In [15]:
def load_and_preprocess_data(file_path):
    df = pd.read_csv(
        file_path,
        sep=';',
        na_values=['?'],
        low_memory=False
    )

    # Datetime conversion
    df['datetime'] = pd.to_datetime(
        df['Date'] + ' ' + df['Time'],
        format='%d/%m/%Y %H:%M:%S'
    )

    df.set_index('datetime', inplace=True)
    df.sort_index(inplace=True)

    # Drop unused columns
    df.drop(['Date', 'Time'], axis=1, inplace=True)

    # Convert to numeric
    df = df.apply(pd.to_numeric)

    # Handle missing values
    df.interpolate(method='time', inplace=True)

    # Resample to hourly
    energy_hourly = df['Global_active_power'].resample('h').mean()
    energy_hourly.interpolate(method='time', inplace=True)

    return energy_hourly

### 3. Load and Preprocess Data

In [16]:
energy_hourly = load_and_preprocess_data('../data/raw/household_power_consumption.txt')
energy_hourly.head()

datetime
2006-12-16 17:00:00    4.222889
2006-12-16 18:00:00    3.632200
2006-12-16 19:00:00    3.400233
2006-12-16 20:00:00    3.268567
2006-12-16 21:00:00    3.056467
Freq: h, Name: Global_active_power, dtype: float64

In [17]:
# Export processed data

os.makedirs('../data/processed', exist_ok=True)

energy_hourly.to_csv('../data/processed/energy_hourly.csv')

print("Processed data saved successfully.")

Processed data saved successfully.


### Saving Processed Data

The preprocessed dataset is saved to the `data/processed` directory. 

This allows reuse of clean data in downstream modeling steps without repeating preprocessing, improving efficiency and workflow clarity.

### 4. Stationarity Test (ADF Test)

The Augmented Dickey-Fuller (ADF) test is used to determine whether the time series is stationary.

- Null Hypothesis: The series is non-stationary
- Alternative Hypothesis: The series is stationary

In [18]:
result = adfuller(energy_hourly)

print('ADF Statistic:', result[0])
print('p-value:', result[1])

ADF Statistic: -14.371344689221477
p-value: 9.482687472633596e-27


### Interpretation

- The p-value is significantly less than 0.05, which indicates that the null hypothesis can be rejected. Therefore, the time series is stationary.
- This suggests that differencing may not be strictly required for this dataset.

In [19]:
energy_diff = energy_hourly.diff().dropna()

In [20]:
result = adfuller(energy_diff)

print('ADF Statistic:', result[0])
print('p-value:', result[1])

ADF Statistic: -42.81070644158096
p-value: 0.0


### Stationarity Analysis

- The Augmented Dickey-Fuller (ADF) test results indicate that the original time series is already stationary, as the p-value is significantly less than 0.05.
- Although differencing was not strictly required, first-order differencing was applied to further validate stationarity and to ensure compatibility with models like SARIMA.
- After differencing, the series remained stationary with an even lower p-value, confirming the stability of the time series.
- This indicates that the dataset does not exhibit strong non-stationarity, simplifying the modeling process.